# Diseño y preparación de la base de datos y Data Mart

Este notebook documenta el proceso para pasar del dataset procesado a una base de datos relacional y a un Data Mart dimensional. Se alinea con las indicaciones del Proyecto Integrador: documentar la limpieza y el ETL, y construir un Data Mart con tabla de hechos, dimensiones, llaves y medidas.

**Motor previsto:** Microsoft SQL Server, sujeto a confirmar en el equipo donde se ejecutará el proyecto.

**Fuente de carga:** `data/processed/clean/online_shoppers_intention_clean.csv`. El archivo original en `data/raw/` se conserva sin cambios.

**Procedimiento de carga:** Los datos se cargan a SQL Server mediante Python (`src/etl/cargar_sql.py`) usando consultas parametrizadas, no mediante `BULK INSERT` ni SQL puro. El script SQL `sql/dml/02_cargar_dataset.sql` solo valida la carga.

> **Nota importante:** La implementación actual del ETL en Python es **funcional y operativa** para desarrollo y validación, pero **no es la implementación definitiva**. El ETL final de producción se desarrollará en **Visual Studio (SSIS)**. Tampoco el dashboard actual en Streamlit es el definitivo; la visualización final se realizará en **Power BI** conectado directamente al Data Mart.

## 1. Estado y objetivo

El dataset procesado disponible contiene 12,205 sesiones y 18 variables. En la revisión local se observaron cero duplicados exactos y cero valores faltantes. La variable `Month` está estandarizada con abreviaturas; por ejemplo, `June` se transformó en `Jun`.

El objetivo de esta etapa es almacenar las sesiones y sus atributos para consultar el comportamiento de compra. La base tendrá una zona de staging para recibir el CSV y un esquema estrella para el Data Mart.

### 1.1 Diccionario de variables del dataset (18 variables)

La siguiente tabla documenta las 18 variables del dataset procesado con su descripción, tipo estadístico, escala, tipo de almacenamiento propuesto en SQL Server y justificación.

| Nombre de la variable | Descripción | Tipo estadístico | Escala / Clasificación | Tipo SQL Server (propuesto) | Justificación del tipo |
|---|---|---|---|---|---|
| `Administrative` | Número de páginas administrativas visitadas | Cuantitativa discreta | Ratio (cuenta) | `INT` | Entero no negativo; cuenta de páginas |
| `Administrative_Duration` | Tiempo total en páginas administrativas (segundos) | Cuantitativa continua | Ratio (duración) | `DECIMAL(14,6)` | Precisión para segundos con decimales; valores grandes |
| `Informational` | Número de páginas informativas visitadas | Cuantitativa discreta | Ratio (cuenta) | `INT` | Entero no negativo; cuenta de páginas |
| `Informational_Duration` | Tiempo total en páginas informativas (segundos) | Cuantitativa continua | Ratio (duración) | `DECIMAL(14,6)` | Precisión para segundos con decimales; valores grandes |
| `ProductRelated` | Número de páginas de producto visitadas | Cuantitativa discreta | Ratio (cuenta) | `INT` | Entero no negativo; cuenta de páginas |
| `ProductRelated_Duration` | Tiempo total en páginas de producto (segundos) | Cuantitativa continua | Ratio (duración) | `DECIMAL(14,6)` | Precisión para segundos con decimales; variable objetivo en regresión |
| `BounceRates` | Tasa de rebote de la sesión | Cuantitativa continua | Ratio (proporción [0,1]) | `DECIMAL(12,8)` | Alta precisión para tasas; 8 decimales cubren granularidad |
| `ExitRates` | Tasa de salida de la sesión | Cuantitativa continua | Ratio (proporción [0,1]) | `DECIMAL(12,8)` | Alta precisión para tasas; 8 decimales cubren granularidad |
| `PageValues` | Valor medio de página previo a compra | Cuantitativa continua | Ratio (valor monetario/índice ≥ 0) | `DECIMAL(14,6)` | Soporta valores grandes y decimales; medida de valor |
| `SpecialDay` | Proximidad a día festivo (0 = lejos, 1 = día festivo) | Cuantitativa continua | Ratio (índice [0,1]) | `DECIMAL(8,6)` | Rango acotado [0,1]; 6 decimales suficiente |
| `Month` | Mes de la sesión (abreviatura 3 letras) | Cualitativa | Nominal (10 categorías: Feb–Dec) | `NVARCHAR(3)` | Texto corto fijo; abreviatura estándar (ej. `Feb`, `Mar`) |
| `OperatingSystems` | Código de sistema operativo (1–8) | Cualitativa | Nominal (códigos 1–8) | `INT` (en staging) → `INT` en `DimSistemaOperativo` | Código categórico numérico; no es cantidad |
| `Browser` | Código de navegador (1–13) | Cualitativa | Nominal (códigos 1–13) | `INT` (en staging) → `INT` en `DimNavegador` | Código categórico numérico; no es cantidad |
| `Region` | Código de región (1–9) | Cualitativa | Nominal (códigos 1–9) | `INT` (en staging) → `INT` en `DimRegion` | Código categórico numérico; no es cantidad |
| `TrafficType` | Código de tipo de tráfico (1–20) | Cualitativa | Nominal (códigos 1–20) | `INT` (en staging) → `INT` en `DimTipoTrafico` | Código categórico numérico; no es cantidad |
| `VisitorType` | Tipo de visitante | Cualitativa | Nominal (3 categorías: `New_Visitor`, `Returning_Visitor`, `Other`) | `NVARCHAR(30)` (en staging) → `NVARCHAR(30)` en `DimTipoVisitante` | Texto descriptivo; longitud máxima conocida |
| `Weekend` | Si la sesión fue en fin de semana | Cualitativa | Binaria / Dícotoma (`True`/`False`) | `NVARCHAR(5)` (en staging) → `BIT` en `FactSesion` | Indicador booleano; `BIT` óptimo en hecho |
| `Revenue` | Si la sesión terminó en compra | Cualitativa | Binaria / Dícotoma (`True`/`False`) | `NVARCHAR(5)` (en staging) → `BIT` en `FactSesion` | Variable objetivo clasificación; indicador booleano |

> **Nota sobre los tipos SQL:** Los tipos propuestos para las tablas definitivas (`Dim*` y `FactSesion`) coinciden con el script `sql/ddl/01_crear_base_y_datamart.sql`. En `StgSesiones` se usan `NVARCHAR` para `BounceRates`, `Weekend` y `Revenue` porque el CSV trae estos valores como texto (`"True"/"False"` y números en formato string); el ETL convierte a `DECIMAL`/`BIT` al cargar el hecho.

In [3]:
from pathlib import Path
import pandas as pd

# Detectar raíz del proyecto de forma robusta
raiz = Path.cwd()
while raiz.name and raiz.name != 'REC_ECDB_2026_ChavezMolina_Byron':
    if raiz.parent == raiz:
        break
    raiz = raiz.parent

ruta_csv = raiz / 'data' / 'processed' / 'clean' / 'online_shoppers_intention_clean.csv'
datos = pd.read_csv(ruta_csv)

print(f'Filas: {datos.shape[0]:,}')
print(f'Columnas: {datos.shape[1]}')
print(f'Duplicados exactos: {datos.duplicated().sum():,}')
print(f'Valores faltantes: {datos.isna().sum().sum():,}')
print('Columnas:', datos.columns.tolist())

Filas: 12,205
Columnas: 18
Duplicados exactos: 0
Valores faltantes: 0
Columnas: ['Administrative', 'Administrative_Duration', 'Informational', 'Informational_Duration', 'ProductRelated', 'ProductRelated_Duration', 'BounceRates', 'ExitRates', 'PageValues', 'SpecialDay', 'Month', 'OperatingSystems', 'Browser', 'Region', 'TrafficType', 'VisitorType', 'Weekend', 'Revenue']


## 2. Grano y reglas del modelo

**Grano de `FactSesion`: una fila por cada sesión del dataset procesado.** El archivo fuente no incluye un identificador único de sesión, por lo que SQL Server generará una llave sustituta (`SesionKey`) para identificar cada fila dentro del Data Mart. Esa llave no representa un identificador original del sitio web.

> **Aclaración importante:** El dataset original crudo tenía 12,330 filas. Durante la limpieza (notebook `02_Limpieza de datos.ipynb`) se eliminaron **125 duplicados exactos** (filas idénticas en las 18 columnas), resultando en **12,205 filas** en el dataset procesado. Los duplicados exactos eliminados **no necesariamente representan la misma sesión real**; son filas idénticas en todas las variables observadas, lo que sugiere un posible error de captura o registro doble, pero no se puede afirmar con certeza que sean la misma sesión física del usuario. Por ello, la decisión conservadora fue eliminar duplicados exactos manteniendo la primera ocurrencia.

Las variables categóricas se separan en dimensiones para facilitar filtros y agrupaciones. Algunas se almacenan como números en el CSV, pero representan códigos, no cantidades. Las variables de páginas, duración y tasas se conservan como medidas. `Revenue` se conserva como indicador de compra y variable objetivo para clasificación; `Weekend` se conserva como indicador de sesión.

### 2.1 Variables utilizadas en cada modelo analítico

Los tres modelos (regresión, clasificación y clustering) usan subconjuntos distintos de variables del dataset. A continuación se documenta claramente qué variables usa cada uno:

#### Regresión (Unidad III — `03_modelos_supervisados.ipynb`)
- **Objetivo (target):** `ProductRelated_Duration` — duración total en páginas de producto (segundos).
- **Features (predictoras):** todas las demás variables **excepto** `Revenue` y `ProductRelated_Duration` (16 variables predictoras).
- **Razón:** se busca estimar el tiempo de navegación en producto sin usar el resultado de compra como predictor (evita *data leakage* y tiene sentido de negocio: predecir duración antes de saber si compra). Tampoco se usa la propia variable objetivo como predictora.

#### Clasificación (Unidad III — `03_modelos_supervisados.ipynb`)
- **Objetivo (target):** `Revenue` — indicador binario de compra (1 = compra, 0 = no compra).
- **Features (predictoras):** las 17 variables restantes (`Month`, `OperatingSystems`, `Browser`, `Region`, `TrafficType`, `VisitorType`, `Weekend`, `Administrative`, `Administrative_Duration`, `Informational`, `Informational_Duration`, `ProductRelated`, `ProductRelated_Duration`, `BounceRates`, `ExitRates`, `PageValues`, `SpecialDay`).
- **Razón:** predecir si una sesión terminará en compra usando el comportamiento de navegación y contexto.

#### Clustering (Unidad IV — `04_clustering_pca.ipynb`)
- **Variables de entrada (10 variables conductuales numéricas):**
  1. `Administrative`
  2. `Administrative_Duration`
  3. `Informational`
  4. `Informational_Duration`
  5. `ProductRelated`
  6. `ProductRelated_Duration`
  7. `BounceRates`
  8. `ExitRates`
  9. `PageValues`
  10. `SpecialDay`
- **Variables EXCLUIDAS del clustering:**
  - `Month`, `OperatingSystems`, `Browser`, `Region`, `TrafficType` → son códigos categóricos nominales, no variables conductuales continuas.
  - `VisitorType` → categórica nominal.
  - `Weekend` → binaria, no continua.
  - `Revenue` → **variable objetivo de clasificación; NO se usa para entrenar el clustering**.
- **Uso de `Revenue` en clustering:** `Revenue` **no participa en el ajuste de K-Means**. Solo se consulta **a posteriori** para describir y comparar los grupos resultantes (tasa de compra por cluster). Esto permite interpretar los clusters en términos de propensión a compra sin sesgar la formación de los grupos hacia la etiqueta de compra.
- **Razón:** el clustering busca agrupar sesiones por **patrones de navegación y comportamiento** (páginas vistas, tiempo, tasas, valor de página, proximidad a festivos), no por características técnicas (navegador, SO, región) ni por el resultado final de compra.

## 3. Definición del grano y diseño dimensional del Data Mart

### 3.1 Qué significa el grano

El **grano** (o *grain*) de una tabla de hechos define **qué representa una sola fila** en esa tabla. En este Data Mart, el grano de `FactSesion` es:

> **Una fila por cada registro del dataset procesado.**

Esto significa que cada registro en `FactSesion` corresponde a una fila del CSV limpio (12,205 filas), independientemente de si esa visita terminó en compra (`Revenue = 1`) o no (`Revenue = 0`). El archivo fuente no incluye un identificador único de sesión que permita garantizar la unicidad de sesión real; por tanto, cada fila del dataset procesado se carga como un registro de hecho. `FactSesion` tendrá exactamente 12,205 filas tras la carga.

### 3.2 Por qué se eligió este nivel de detalle

El análisis del comportamiento de compradores en comercio electrónico requiere entender **qué ocurre durante una visita concreta**:

- **Navegación:** número y tipo de páginas visitadas (administrativas, informativas, de producto).
- **Tiempo:** duración en cada tipo de página.
- **Calidad de la sesión:** tasas de rebote (`BounceRates`), salida (`ExitRates`), valor de página (`PageValues`).
- **Contexto temporal:** mes (`Month`), día de fin de semana (`Weekend`), proximidad a día especial (`SpecialDay`).
- **Entorno técnico:** sistema operativo, navegador, región, tipo de tráfico.
- **Perfil del visitante:** nuevo, recurrente u otro (`VisitorType`).
- **Resultado:** ¿hubo compra? (`Revenue`).

Una granularidad más fina (p. ej., a nivel de *pageview* o *click*) no existe en el dataset original. Una granularidad más gruesa (p. ej., agregado por día o por visitante) perdería la capacidad de comparar sesiones individuales con y sin compra, que es el objetivo central del modelado supervisado (clasificación) y del análisis no supervisado (clustering).

### 3.3 Qué representa una fila de `FactSesion`

| Componente | Descripción |
|------------|-------------|
| **Llaves foráneas** (6) | Conectan la sesión con sus dimensiones: `MesKey`, `TipoVisitanteKey`, `SistemaOperativoKey`, `NavegadorKey`, `RegionKey`, `TipoTraficoKey`. |
| **Medidas de actividad** (6) | `Administrative`, `Administrative_Duration`, `Informational`, `Informational_Duration`, `ProductRelated`, `ProductRelated_Duration`. |
| **Tasas y valor** (3) | `BounceRates`, `ExitRates`, `PageValues`. |
| **Indicadores contextuales** (2) | `SpecialDay` (proximidad a día festivo), `Weekend` (sesión en fin de semana). |
| **Variable objetivo** | `Revenue` (1 = compra, 0 = no compra). |

### 3.4 Qué representa la clave primaria `SesionKey`

`SesionKey` es una **llave sustituta** (*surrogate key*) de tipo `BIGINT IDENTITY(1,1)` generada automáticamente por SQL Server al insertar cada fila en `FactSesion`.

- **No** proviene del dataset original (el CSV no tiene identificador de sesión).
- **No** tiene significado de negocio; solo sirve para identificar unívocamente la fila dentro del Data Mart.
- Permite referenciar la fact desde otras tablas si fuera necesario (p. ej., una tabla de hechos de detalle de clics futura).

### 3.5 Relación de cada sesión con las seis dimensiones

Cada sesión se relaciona **exactamente con una** fila de cada dimensión:

| Dimensión | Columna natural en CSV | Llave en `FactSesion` | Cardinalidad |
|-----------|------------------------|------------------------|--------------|
| `DimMes` | `Month` (ej. `Feb`, `Mar`) | `MesKey` | N:1 |
| `DimTipoVisitante` | `VisitorType` (ej. `Returning_Visitor`) | `TipoVisitanteKey` | N:1 |
| `DimSistemaOperativo` | `OperatingSystems` (código 1–8) | `SistemaOperativoKey` | N:1 |
| `DimNavegador` | `Browser` (código 1–13) | `NavegadorKey` | N:1 |
| `DimRegion` | `Region` (código 1–9) | `RegionKey` | N:1 |
| `DimTipoTrafico` | `TrafficType` (código 1–20) | `TipoTraficoKey` | N:1 |

Todas las dimensiones usan **llaves sustitutas enteras** como PK y tienen restricción `UNIQUE` sobre su atributo natural, lo que garantiza que no haya duplicados y soporta el `MERGE` upsert en la carga.

### 3.6 Tabla de staging (`StgSesiones`)

La tabla de staging recibe el CSV procesado tal cual, conservando una fila por sesión y los tipos de datos que facilitan la carga inicial desde Python:

```sql
CREATE TABLE dbo.StgSesiones (
    Administrative INT NOT NULL,
    Administrative_Duration DECIMAL(14, 6) NOT NULL,
    Informational INT NOT NULL,
    Informational_Duration DECIMAL(14, 6) NOT NULL,
    ProductRelated INT NOT NULL,
    ProductRelated_Duration DECIMAL(14, 6) NOT NULL,
    BounceRates NVARCHAR(32) NOT NULL,
    ExitRates DECIMAL(12, 8) NOT NULL,
    PageValues DECIMAL(14, 6) NOT NULL,
    SpecialDay DECIMAL(8, 6) NOT NULL,
    [Month] NVARCHAR(3) NOT NULL,
    OperatingSystems INT NOT NULL,
    Browser INT NOT NULL,
    Region INT NOT NULL,
    TrafficType INT NOT NULL,
    VisitorType NVARCHAR(30) NOT NULL,
    Weekend NVARCHAR(5) NOT NULL,
    Revenue NVARCHAR(5) NOT NULL
);
```

**Notas sobre `StgSesiones`:**
- `BounceRates` se guarda como `NVARCHAR(32)` porque en el CSV viene como texto numérico (ej. `"0.200000"`), y el ETL lo castea a `DECIMAL(12,8)` al cargar el hecho.
- `Weekend` y `Revenue` se guardan como `NVARCHAR(5)` porque el CSV trae `"True"`/`"False"`; el ETL los convierte a `BIT` (0/1) en `FactSesion`.
- No tiene llave primaria ni restricciones únicas; es una tabla de carga transitoria que se trunca (`TRUNCATE`) en cada ejecución del ETL.

### 3.7 Seis tablas de dimensión

Cada dimensión tiene una **llave sustituta** (`*Key INT IDENTITY(1,1) PRIMARY KEY`) y una restricción `UNIQUE` sobre su atributo natural (el valor que viene del CSV). Esto permite el `MERGE` upsert idempotente en el ETL.

```sql
-- DimMes
CREATE TABLE dbo.DimMes (
    MesKey INT IDENTITY(1,1) NOT NULL CONSTRAINT PK_DimMes PRIMARY KEY,
    Mes NVARCHAR(3) NOT NULL CONSTRAINT UQ_DimMes_Mes UNIQUE
);

-- DimTipoVisitante
CREATE TABLE dbo.DimTipoVisitante (
    TipoVisitanteKey INT IDENTITY(1,1) NOT NULL CONSTRAINT PK_DimTipoVisitante PRIMARY KEY,
    TipoVisitante NVARCHAR(30) NOT NULL CONSTRAINT UQ_DimTipoVisitante UNIQUE
);

-- DimSistemaOperativo
CREATE TABLE dbo.DimSistemaOperativo (
    SistemaOperativoKey INT IDENTITY(1,1) NOT NULL CONSTRAINT PK_DimSistemaOperativo PRIMARY KEY,
    CodigoSistemaOperativo INT NOT NULL CONSTRAINT UQ_DimSistemaOperativo UNIQUE
);

-- DimNavegador
CREATE TABLE dbo.DimNavegador (
    NavegadorKey INT IDENTITY(1,1) NOT NULL CONSTRAINT PK_DimNavegador PRIMARY KEY,
    CodigoNavegador INT NOT NULL CONSTRAINT UQ_DimNavegador UNIQUE
);

-- DimRegion
CREATE TABLE dbo.DimRegion (
    RegionKey INT IDENTITY(1,1) NOT NULL CONSTRAINT PK_DimRegion PRIMARY KEY,
    CodigoRegion INT NOT NULL CONSTRAINT UQ_DimRegion UNIQUE
);

-- DimTipoTrafico
CREATE TABLE dbo.DimTipoTrafico (
    TipoTraficoKey INT IDENTITY(1,1) NOT NULL CONSTRAINT PK_DimTipoTrafico PRIMARY KEY,
    CodigoTipoTrafico INT NOT NULL CONSTRAINT UQ_DimTipoTrafico UNIQUE
);
```

### 3.8 Tabla de hechos (`FactSesion`)

```sql
CREATE TABLE dbo.FactSesion (
    SesionKey BIGINT IDENTITY(1,1) NOT NULL CONSTRAINT PK_FactSesion PRIMARY KEY,
    -- Llaves foráneas (6 dimensiones)
    MesKey INT NOT NULL,
    TipoVisitanteKey INT NOT NULL,
    SistemaOperativoKey INT NOT NULL,
    NavegadorKey INT NOT NULL,
    RegionKey INT NOT NULL,
    TipoTraficoKey INT NOT NULL,
    -- Medidas de navegación
    Administrative INT NOT NULL,
    Administrative_Duration DECIMAL(14, 6) NOT NULL,
    Informational INT NOT NULL,
    Informational_Duration DECIMAL(14, 6) NOT NULL,
    ProductRelated INT NOT NULL,
    ProductRelated_Duration DECIMAL(14, 6) NOT NULL,
    -- Tasas y valor
    BounceRates DECIMAL(12, 8) NOT NULL,
    ExitRates DECIMAL(12, 8) NOT NULL,
    PageValues DECIMAL(14, 6) NOT NULL,
    -- Indicadores
    SpecialDay DECIMAL(8, 6) NOT NULL,
    Weekend BIT NOT NULL,      -- 1 = fin de semana, 0 = entre semana
    Revenue BIT NOT NULL,      -- 1 = compra, 0 = no compra
    -- Restricciones de integridad referencial
    CONSTRAINT FK_FactSesion_DimMes FOREIGN KEY (MesKey) REFERENCES dbo.DimMes(MesKey),
    CONSTRAINT FK_FactSesion_DimTipoVisitante FOREIGN KEY (TipoVisitanteKey) REFERENCES dbo.DimTipoVisitante(TipoVisitanteKey),
    CONSTRAINT FK_FactSesion_DimSistemaOperativo FOREIGN KEY (SistemaOperativoKey) REFERENCES dbo.DimSistemaOperativo(SistemaOperativoKey),
    CONSTRAINT FK_FactSesion_DimNavegador FOREIGN KEY (NavegadorKey) REFERENCES dbo.DimNavegador(NavegadorKey),
    CONSTRAINT FK_FactSesion_DimRegion FOREIGN KEY (RegionKey) REFERENCES dbo.DimRegion(RegionKey),
    CONSTRAINT FK_FactSesion_DimTipoTrafico FOREIGN KEY (TipoTraficoKey) REFERENCES dbo.DimTipoTrafico(TipoTraficoKey)
);
```

**Medidas e indicadores en `FactSesion`:**
- **Medidas aditivas (cuentas):** `Administrative`, `Informational`, `ProductRelated` — se pueden sumar, promediar, contar.
- **Medidas aditivas (duraciones):** `Administrative_Duration`, `Informational_Duration`, `ProductRelated_Duration` — sumables, promediables.
- **Medidas no aditivas (tasas/índices):** `BounceRates`, `ExitRates`, `PageValues`, `SpecialDay` — promediables, pero no sumables con sentido de negocio directo.
- **Indicadores binarios:** `Weekend`, `Revenue` — filtrables, contables (SUM = cantidad de sesiones con compra / fin de semana).

### 3.9 Cómo se identifica una compra mediante `Revenue`

`Revenue` es un indicador binario (`BIT` en SQL Server, `0`/`1` en Python):

- `Revenue = 1` → la sesión **terminó en compra** (el visitante realizó al menos una transacción).
- `Revenue = 0` → la sesión **no terminó en compra** (navegación sin transacción).

Es la **variable objetivo** para:
- **Clasificación supervisada:** predecir si una sesión nueva comprará.
- **Análisis descriptivo:** comparar comportamiento (páginas, tiempo, tasas) entre compradores y no compradores.
- **Segmentación no supervisada:** los clusters de K-Means pueden interpretarse según su tasa de `Revenue`.

### 3.10 Justificación del diseño dimensional

#### Análisis a nivel de sesión

El grano de sesión permite:

1. **Analizar comportamiento de navegación** sesión a sesión: cuántas páginas de cada tipo, cuánto tiempo, qué tasas de rebote/salida.
2. **Comparar sesiones con compra vs. sin compra** directamente, fila a fila, sin agregaciones que oculten variabilidad.
3. **Estudiar el comportamiento por contexto** usando las dimensiones: ¿compran más los `Returning_Visitor` en `Nov`? ¿Qué `TrafficType` trae sesiones con mayor `PageValues`?
4. **Alimentar modelos de Machine Learning** tal como vienen del dataset original:
   - **Regresión:** `ProductRelated_Duration` como target, features de la misma fila.
   - **Clasificación:** `Revenue` como target, resto de columnas como features.
   - **Clustering:** las 10 variables conductuales numéricas de la fila para agrupar sesiones similares.

#### Esquema estrella (relaciones)

```mermaid
erDiagram
    DimMes ||--o{ FactSesion : "MesKey"
    DimTipoVisitante ||--o{ FactSesion : "TipoVisitanteKey"
    DimSistemaOperativo ||--o{ FactSesion : "SistemaOperativoKey"
    DimNavegador ||--o{ FactSesion : "NavegadorKey"
    DimRegion ||--o{ FactSesion : "RegionKey"
    DimTipoTrafico ||--o{ FactSesion : "TipoTraficoKey"

    FactSesion {
        BIGINT SesionKey PK
        INT MesKey FK
        INT TipoVisitanteKey FK
        INT SistemaOperativoKey FK
        INT NavegadorKey FK
        INT RegionKey FK
        INT TipoTraficoKey FK
        INT Administrative
        DECIMAL Administrative_Duration
        INT Informational
        DECIMAL Informational_Duration
        INT ProductRelated
        DECIMAL ProductRelated_Duration
        DECIMAL BounceRates
        DECIMAL ExitRates
        DECIMAL PageValues
        DECIMAL SpecialDay
        BIT Weekend
        BIT Revenue
    }

    DimMes {
        INT MesKey PK
        NVARCHAR(3) Mes UK
    }
    DimTipoVisitante {
        INT TipoVisitanteKey PK
        NVARCHAR(30) TipoVisitante UK
    }
    DimSistemaOperativo {
        INT SistemaOperativoKey PK
        INT CodigoSistemaOperativo UK
    }
    DimNavegador {
        INT NavegadorKey PK
        INT CodigoNavegador UK
    }
    DimRegion {
        INT RegionKey PK
        INT CodigoRegion UK
    }
    DimTipoTrafico {
        INT TipoTraficoKey PK
        INT CodigoTipoTrafico UK
    }
```

#### Verificación contra DDL y CSV

| Comprobación | Resultado |
|--------------|-----------|
| `FactSesion` tiene 1 fila por sesión del CSV (12,205) | ✅ El ETL inserta una fila por `iterrows()` del DataFrame limpio. |
| `SesionKey` es llave sustituta `IDENTITY`, no viene del CSV | ✅ DDL: `SesionKey BIGINT IDENTITY(1,1) PRIMARY KEY`. |
| 6 dimensiones con PK sustituta y UK en atributo natural | ✅ DDL: cada dimensión tiene `*Key IDENTITY PK` y `UNIQUE` en columna natural. |
| FK en `FactSesion` referencian PK de dimensiones | ✅ DDL: 6 `FOREIGN KEY` constraints declaradas. |
| `Weekend` y `Revenue` como `BIT` (0/1) en hecho, `NVARCHAR` en staging | ✅ Staging guarda 'True'/'False'; ETL convierte con `validar_bool_str()`. |
| `BounceRates` como `NVARCHAR` en staging, `DECIMAL` en hecho | ✅ Staging: `NVARCHAR(32)`; Hecho: `DECIMAL(12,8)`; ETL castea a `float`. |
| Medidas numéricas conservan precisión del CSV | ✅ `DECIMAL(14,6)` para duraciones, `DECIMAL(12,8)` para tasas. |
| Carga idempotente (re-ejecutable sin duplicados) | ✅ `TRUNCATE` staging/hecho + `MERGE` en dimensiones. |

> **Conclusión:** La implementación respeta estrictamente el grano definido. Cada sesión del CSV limpio genera una y solo una fila en `FactSesion`, vinculada a sus seis dimensiones mediante llaves sustitutas, con todas las medidas e indicadores necesarios para el análisis descriptivo, predictivo y de segmentación.

## 4. Mapeo de variables a tablas

| Tabla | Variables | Uso |
|---|---|---|
| `StgSesiones` | Las 18 columnas del CSV | Recibir la carga procesada y conservar una fila fuente por sesión. |
| `DimMes` | `Month` | Filtrar y agrupar por mes. |
| `DimTipoVisitante` | `VisitorType` | Analizar visitantes nuevos, recurrentes y otros. |
| `DimSistemaOperativo` | `OperatingSystems` | Analizar el código de sistema operativo. |
| `DimNavegador` | `Browser` | Analizar el código de navegador. |
| `DimRegion` | `Region` | Analizar el código de región. |
| `DimTipoTrafico` | `TrafficType` | Analizar el código de tipo de tráfico. |
| `FactSesion` | Llaves a dimensiones; cantidades de páginas; duraciones; tasas; `PageValues`; `SpecialDay`; `Weekend`; `Revenue` | Mantener una fila por sesión y sus medidas/indicadores. |

Las dimensiones tendrán llaves sustitutas enteras como PK. `FactSesion` guardará esas llaves como FK y tendrá su propia llave sustituta (`SesionKey`) generada por SQL Server. La carga reconstruye la tabla de hechos a partir de staging en cada ejecución.

### 4.1 Uso de las tablas en Power BI

Power BI se conectará directamente al Data Mart en SQL Server y **utilizará únicamente las tablas necesarias para cada análisis** (p. ej., `FactSesion` + `DimMes` + `DimTipoVisitante` para análisis de compra por mes y tipo de visitante). **No se eliminan dimensiones del Data Mart** por no usarse en un informe concreto; el Data Mart conserva todas las dimensiones para dar servicio a cualquier análisis futuro. La selección de tablas en Power BI es decisión del analista al diseñar el modelo semántico, no una restricción física del Data Mart.

## 5. Orden del proceso

1. Crear la base `ECBD_Compradores` en SQL Server.
2. Crear `StgSesiones` con las 18 variables del CSV, conservando una fila por sesión.
3. Ejecutar el ETL de Python que lee el CSV, limpia los datos y los carga a SQL Server:
   - `python -m src.etl.etl --cargar-sql`
   > **Nota:** Este ETL en Python es la implementación **actual de desarrollo y validación**. El ETL final de producción se implementará en **Visual Studio (SSIS)**.
4. Validar la carga con el script `sql/dml/02_cargar_dataset.sql` (solo validación).

### Paso 1: crear la base de datos

Ejecutar en una consulta de SQL Server Management Studio conectada al servidor. `USE` selecciona la base para las instrucciones posteriores.

La creación de la base de datos, la tabla de staging, las dimensiones y la tabla de hechos se encuentra implementada en el script `sql/ddl/01_crear_base_y_datamart.sql`.

La carga del dataset procesado se realiza desde Python ejecutando:
```powershell
.venv\Scripts\python.exe -m src.etl.etl --cargar-sql
```

> **Importante:** La carga vía Python (`src/etl/cargar_sql.py`) es operativa para desarrollo y pruebas. El ETL definitivo se desarrollará en **Visual Studio (SSIS)** con paquetes de Integration Services. La visualización final no será en Streamlit sino en **Power BI** conectado directamente al Data Mart.

La validación de la carga se realiza con `sql/dml/02_cargar_dataset.sql`, que verifica conteos, integridad referencial y muestra una muestra de datos.

## 5. Validaciones de carga previstas

Al terminar el ETL, comparar el número de filas de staging y de `FactSesion`. Como el dataset procesado tiene una fila por sesión y las dimensiones se construyen con valores distintos, la carga del hecho debe conservar las 12,205 sesiones. También se comprobará que no haya llaves foráneas sin correspondencia y que las dimensiones no tengan duplicados en sus atributos naturales.

Consultas de control incluidas en el script `sql/dml/02_cargar_dataset.sql` para revisar los registros cargados:

```sql
SELECT COUNT(*) AS FilasStaging FROM dbo.StgSesiones;
SELECT COUNT(*) AS FilasHechos FROM dbo.FactSesion;
SELECT TOP (5) * FROM dbo.FactSesion;
```

## 7. Estado del proyecto

Esta sección distingue claramente entre lo **implementado** (código ejecutable y probado), lo **documentado** (diseño, decisiones, especificaciones en este notebook) y lo **pendiente** (trabajo por hacer en siguientes fases).

### 7.1 Implementado (código funcional y probado)

| Componente | Estado | Ubicación | Observaciones |
|---|---|---|---|
| Dataset procesado limpio (12,205 filas, 18 variables) | ✅ Implementado | `data/processed/clean/online_shoppers_intention_clean.csv` | Validado: 0 duplicados, 0 nulos, meses estandarizados |
| Script DDL: creación de base, staging, 6 dimensiones, hecho | ✅ Implementado | `sql/ddl/01_crear_base_y_datamart.sql` | Probado en SQL Server LocalDB |
| ETL Python (carga parametrizada, MERGE en dimensiones, truncate/insert en hecho) | ✅ Implementado | `src/etl/cargar_sql.py`, `src/etl/etl.py` | Funcional para desarrollo y validación |
| Script de validación de carga (conteos, FK, muestra) | ✅ Implementado | `sql/dml/02_cargar_dataset.sql` | Verifica integridad referencial |
| Modelos supervisados (regresión y clasificación) | ✅ Implementado | `notebooks/unidad_iii_supervisado/03_modelos_supervisados.ipynb` | Pipelines guardados en `modelos/` |
| Clustering K-Means + PCA (10 variables conductuales) | ✅ Implementado | `notebooks/unidad_iv_no_supervisado/04_clustering_pca.ipynb` | Modelos y métricas guardados |
| Dashboard exploratorio en Streamlit | ✅ Implementado | `dashboard/app.py` | Solo para exploración; **no es el dashboard final** |

### 7.2 Documentado (diseño y decisiones en este notebook)

| Tema | Estado | Detalle |
|---|---|---|
| Diccionario de 18 variables (tipo estadístico, escala, tipo SQL, justificación) | ✅ Documentado | Sección 1.1 |
| Grano de `FactSesion` y reglas del modelo | ✅ Documentado | Secciones 2 y 3 |
| Variables usadas en regresión, clasificación y clustering | ✅ Documentado | Sección 2.1 |
| Uso de `Revenue` en clustering (solo describir, no entrenar) | ✅ Documentado | Sección 2.1 |
| Tabla de staging (`StgSesiones`) | ✅ Documentado | Sección 3.6 |
| Seis dimensiones con PK sustituta y UK en atributo natural | ✅ Documentado | Sección 3.7 |
| Tabla de hechos (`FactSesion`) con 6 FK, medidas e indicadores | ✅ Documentado | Sección 3.8 |
| Llaves primarias, foráneas, restricciones UNIQUE | ✅ Documentado | Secciones 3.7, 3.8, 3.10 |
| Medidas aditivas, no aditivas, indicadores binarios | ✅ Documentado | Sección 3.8 |
| Esquema estrella (diagrama ER Mermaid) | ✅ Documentado | Sección 3.10 |
| Verificación de tipos SQL contra DDL existente | ✅ Documentado | Sección 3.10 (tabla) |
| Uso de tablas en Power BI (selección, no eliminación) | ✅ Documentado | Sección 4.1 |
| Aclaración sobre 125 duplicados exactos eliminados en limpieza | ✅ Documentado | Sección 2 |
| `SesionKey` como llave sustituta (no identificador original) | ✅ Documentado | Secciones 2, 3.4 |

### 7.3 Pendiente (próximos pasos del proyecto)

| Componente | Estado | Responsable / Herramienta | Comentario |
|---|---|---|---|
| ETL de producción en **Visual Studio (SSIS)** | ⏳ Pendiente | Equipo / VS + SSIS | Reemplazará al ETL Python actual |
| Dashboard final en **Power BI** | ⏳ Pendiente | Equipo / Power BI Desktop + Service | Conectado directo al Data Mart; **no Streamlit** |
| Ejecución de scripts DDL y validación en entorno de destino | ⏳ Pendiente | Equipo / SSMS | Confirmar en el servidor final |
| Pruebas de carga y rendimiento del Data Mart | ⏳ Pendiente | Equipo | Índices, particionamiento si procede |
| Documentación de usuario / manual de Power BI | ⏳ Pendiente | Equipo | Para analistas de negocio |
| Validación cruzada y ajuste de hiperparámetros en modelos ML | ⏳ Pendiente | Ciencia de datos | Actualmente semilla fija 42; falta CV |
| Despliegue en entorno de producción | ⏳ Pendiente | DevOps / Equipo | CI/CD, seguridad, backups |

---

> **Resumen para el profesor:**
> - El **diseño dimensional** (estrella: 1 hecho + 6 dimensiones + staging) está **completamente documentado y verificado contra el DDL**.
> - Los **tipos SQL propuestos coinciden** con `sql/ddl/01_crear_base_y_datamart.sql`.
> - Las **variables de los tres modelos** (regresión, clasificación, clustering) **coinciden** con sus notebooks actuales.
> - El **ETL Python actual es funcional para desarrollo**; el definitivo será **SSIS en Visual Studio**.
> - El **dashboard final será Power BI**; Streamlit es solo exploratorio.
> - **No se eliminan dimensiones** del Data Mart; Power BI usa solo las necesarias por análisis.
> - **125 duplicados exactos** se eliminaron en limpieza (12,330 → 12,205); **no se asume** que sean la misma sesión real.
> - `SesionKey` es **llave sustituta**, no identificador original.
> - **Clustering usa 10 variables conductuales** (no 3); `Revenue` solo describe clusters a posteriori.